Setup and connection

In [4]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings("ignore")

load_dotenv(dotenv_path=Path("../.env"))

CLEANED = Path("../data/cleaned")

password = quote_plus(os.getenv('DB_PASSWORD'))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{password}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}",
    echo=False
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user;"))
    print("Connected to:", result.fetchone())

Connected to: ('olist_ecommerce', 'postgres')


Load all cleaned files

In [8]:
# Step 2: Load all cleaned files
CLEANED = Path("../data/cleaned")

orders = pd.read_csv(CLEANED / "orders_clean.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

order_items = pd.read_csv(CLEANED / "order_items_clean.csv",
                          parse_dates=["shipping_limit_date"])

payments = pd.read_csv(CLEANED / "payments_clean.csv")
reviews = pd.read_csv(CLEANED / "reviews_clean.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])
customers = pd.read_csv(CLEANED / "customers_clean.csv")
sellers = pd.read_csv(CLEANED / "sellers_clean.csv")
products = pd.read_csv(CLEANED / "products_clean.csv")
geo = pd.read_csv(CLEANED / "geolocation_clean.csv")

print("All cleaned files loaded.")
for name, df in [("orders", orders), ("order_items", order_items),
                 ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("sellers", sellers),
                 ("products", products), ("geo", geo)]:
    print(f"  {name}: {df.shape}")

All cleaned files loaded.
  orders: (99441, 15)
  order_items: (112650, 7)
  payments: (99437, 5)
  reviews: (98410, 8)
  customers: (99441, 5)
  sellers: (3095, 4)
  products: (32951, 10)
  geo: (19011, 5)


Rename columns to match schema

In [9]:
# Step 3: Rename columns to match schema
geo = geo.rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})

print("geo columns:", geo.columns.tolist())
print("payments columns:", payments.columns.tolist())

geo columns: ['zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
payments columns: ['order_id', 'payment_value', 'payment_installments', 'payment_type_primary', 'payment_type_count']


Load all tables

In [10]:
# Step 4: Load all tables
def load_table(df, table_name, engine, chunk_size=5000):
    df.to_sql(
        table_name,
        engine,
        schema="olist",
        if_exists="append",
        index=False,
        chunksize=chunk_size,
        method="multi"
    )
    print(f"  Loaded {len(df):,} rows into olist.{table_name}")

print("Loading dimensions...")
load_table(customers, "dim_customers", engine)
load_table(sellers, "dim_sellers", engine)
load_table(products, "dim_products", engine)
load_table(geo, "dim_geolocation", engine)

print("\nLoading fact_orders...")
load_table(orders, "fact_orders", engine)

print("\nLoading fact_order_items...")
load_table(order_items, "fact_order_items", engine)

print("\nLoading bridge tables...")
load_table(payments, "bridge_payments", engine)
load_table(reviews, "bridge_reviews", engine)

print("\nAll tables loaded.")

Loading dimensions...
  Loaded 99,441 rows into olist.dim_customers
  Loaded 3,095 rows into olist.dim_sellers


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "product_name_lenght" of relation "dim_products" does not exist
LINE 1: ....dim_products (product_id, product_category_name, product_na...
                                                             ^

[SQL: INSERT INTO olist.dim_products (product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm, product_category_name_english) VALUES (%(product_id_m0)s, %(product_category_name_m0)s, %(product_name_lenght_m0)s, %(product_description_lenght_m0)s, %(product_photos_qty_m0)s, %(product_weight_g_m0)s, %(product_length_cm_m0)s, %(product_height_cm_m0)s, %(product_width_cm_m0)s, %(product_category_name_english_m0)s), (%(product_id_m1)s, %(product_category_name_m1)s, %(product_name_lenght_m1)s, %(product_description_lenght_m1)s, %(product_photos_qty_m1)s, %(product_weight_g_m1)s, %(product_length_cm_m1)s, %(product_he

In [11]:
# Check actual column names in the products dataframe
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']


In [12]:
# Rename to match schema
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

print("After rename:", products.columns.tolist())

# Update the CSV to keep it consistent
products.to_csv(CLEANED / "products_clean.csv", index=False)
print("products_clean.csv updated")

# Reload just dim_products
load_table(products, "dim_products", engine)

After rename: ['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']
products_clean.csv updated
  Loaded 32,951 rows into olist.dim_products


In [13]:
print("Loading fact_orders...")
load_table(orders, "fact_orders", engine)

print("Loading fact_order_items...")
load_table(order_items, "fact_order_items", engine)

print("Loading bridge tables...")
load_table(payments, "bridge_payments", engine)
load_table(reviews, "bridge_reviews", engine)

print("All tables loaded.")

Loading fact_orders...
  Loaded 99,441 rows into olist.fact_orders
Loading fact_order_items...
  Loaded 112,650 rows into olist.fact_order_items
Loading bridge tables...
  Loaded 99,437 rows into olist.bridge_payments
  Loaded 98,410 rows into olist.bridge_reviews
All tables loaded.


Verify row counts

In [14]:
tables = [
    ("dim_customers", len(customers)),
    ("dim_sellers", len(sellers)),
    ("dim_products", len(products)),
    ("dim_geolocation", len(geo)),
    ("fact_orders", len(orders)),
    ("fact_order_items", len(order_items)),
    ("bridge_payments", len(payments)),
    ("bridge_reviews", len(reviews))
]

print("Row count verification:")
print(f"{'Table':<25} {'Expected':>10} {'Loaded':>10} {'Match':>8}")
print("-" * 58)

with engine.connect() as conn:
    for table, expected in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM olist.{table}"))
        loaded = result.fetchone()[0]
        match = "OK" if loaded == expected else "MISMATCH"
        print(f"{table:<25} {expected:>10,} {loaded:>10,} {match:>8}")

Row count verification:
Table                       Expected     Loaded    Match
----------------------------------------------------------
dim_customers                 99,441     99,441       OK
dim_sellers                    3,095      3,095       OK
dim_products                  32,951     32,951       OK
dim_geolocation               19,011          0 MISMATCH
fact_orders                   99,441     99,441       OK
fact_order_items             112,650    112,650       OK
bridge_payments               99,437     99,437       OK
bridge_reviews                98,410     98,410       OK


End to end join test

In [15]:
test_query = """
SELECT 
    o.order_status,
    COUNT(DISTINCT o.order_id)               AS orders,
    COUNT(DISTINCT c.customer_unique_id)      AS unique_customers,
    ROUND(SUM(p.payment_value)::numeric, 2)  AS total_revenue
FROM olist.fact_orders o
JOIN olist.dim_customers c USING (customer_id)
JOIN olist.bridge_payments p USING (order_id)
GROUP BY o.order_status
ORDER BY total_revenue DESC;
"""

with engine.connect() as conn:
    result = pd.read_sql(test_query, conn)

print(result.to_string(index=False))

order_status  orders  unique_customers  total_revenue
   delivered   96477             93357    15422461.77
     shipped    1107              1100      177213.96
    canceled     622               613      143255.60
 unavailable     609               604      126479.51
  processing     301               301       69394.11
    invoiced     314               314       69137.99
     created       5                 5         688.10
    approved       2                 2         241.08


In [16]:
print("geo shape:", geo.shape)
print("geo columns:", geo.columns.tolist())
print("geo dtypes:")
print(geo.dtypes)
print("\nSample:")
print(geo.head(2))

geo shape: (19011, 5)
geo columns: ['zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
geo dtypes:
zip_code_prefix        int64
geolocation_lat      float64
geolocation_lng      float64
geolocation_city      object
geolocation_state     object
dtype: object

Sample:
   zip_code_prefix  geolocation_lat  geolocation_lng geolocation_city  \
0             1001       -23.549951       -46.634027        sao paulo   
1             1002       -23.548228       -46.635247        sao paulo   

  geolocation_state  
0                SP  
1                SP  


In [17]:
# Check current state in postgres
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM olist.dim_geolocation"))
    print("Current rows in dim_geolocation:", result.fetchone()[0])

# Load it now
load_table(geo, "dim_geolocation", engine)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM olist.dim_geolocation"))
    print("Rows after load:", result.fetchone()[0])

Current rows in dim_geolocation: 0
  Loaded 19,011 rows into olist.dim_geolocation
Rows after load: 19011


In [18]:
tables = [
    ("dim_customers", len(customers)),
    ("dim_sellers", len(sellers)),
    ("dim_products", len(products)),
    ("dim_geolocation", len(geo)),
    ("fact_orders", len(orders)),
    ("fact_order_items", len(order_items)),
    ("bridge_payments", len(payments)),
    ("bridge_reviews", len(reviews))
]

print("Final row count verification:")
print(f"{'Table':<25} {'Expected':>10} {'Loaded':>10} {'Match':>8}")
print("-" * 58)

with engine.connect() as conn:
    for table, expected in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM olist.{table}"))
        loaded = result.fetchone()[0]
        match = "OK" if loaded == expected else "MISMATCH"
        print(f"{table:<25} {expected:>10,} {loaded:>10,} {match:>8}")

Final row count verification:
Table                       Expected     Loaded    Match
----------------------------------------------------------
dim_customers                 99,441     99,441       OK
dim_sellers                    3,095      3,095       OK
dim_products                  32,951     32,951       OK
dim_geolocation               19,011     19,011       OK
fact_orders                   99,441     99,441       OK
fact_order_items             112,650    112,650       OK
bridge_payments               99,437     99,437       OK
bridge_reviews                98,410     98,410       OK
